### TASKS:
	Use this code as a guide for simple image analysis.
	
	1- Update the DFO code to find the brightest area in the input image.
	2- Try the code for other input images
	3- Change input image half-way through and see if your algorithm continues to perform

### ADVANCED TASKS:
	4- Is it possible to trace a moving object? 
	5- How about tracing gradually fading and emerging objects?
	6- What if the input image has two or more bright spots?
	7- If the task was to find symmetry in an image, what would be your fitness function?

#### Load libraries

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import random
from IPython import display
import time
import numpy as np

#### Saving animations
Every animation below is also saved as a GIF in `results/`.

In [ ]:
import os
from PIL import Image
RESULTS = 'results/'
os.makedirs(RESULTS, exist_ok=True)
def grab_frame(gifFrames):
    # SNAPSHOT THE CURRENT FIGURE (IMAGE, FLIES, BEST FLY, ITERATION TITLE) AS ONE GIF FRAME
    fig = plt.gcf()
    fig.canvas.draw()
    gifFrames.append(Image.fromarray(np.asarray(fig.canvas.buffer_rgba())).convert('RGB'))
def save_gif(gifFrames, name, ms=80):
    gifFrames[0].save(RESULTS + name, save_all=True, append_images=gifFrames[1:], duration=ms, loop=0)
    print('Saved animation to', RESULTS + name)

#### Load image and show image width and height

In [ ]:
img = mpimg.imread('input_images/input5.png')
imgW = len(img[0])
imgH = len(img)
print('Image size:', imgW, 'x', imgH)

In [ ]:
# PRINT PIXEL VALUE AT (ROW No, COLUMN No)
print(img[100][360]) # PRINT SAMPLE VALUE AT ROW 100, COLUMN 360
# WHITE PIXEL VALUE = 1.0 (PNG) or 255
# BLACK PIXEL VALUE = 0.0 (PNG) or 0

#### Fitness Function

In [ ]:
def fitness(x) -> int:
    return - img[x[0]][x[1]]

#### Setup

In [ ]:
N = 50                  # Population size (number of flies)
iterations = 100        # Number of iterations
D = 2                   # Dimensionality: [row, column]
delta = 0.001           # Disturbance threshold
lowerB = np.array([0, 0])                  # Lower bounds: first row, first column
upperB = np.array([imgH - 1, imgW - 1])    # Upper bounds: last row, last column
# Initialise flies at random pixel positions within the image
X = np.random.randint(lowerB, upperB + 1, size=(N, D))
fitnesses = np.array([fitness(X[i]) for i in range(N)])
s = np.argmin(fitnesses)    # Index of the swarm's best fly

#### DFO

In [ ]:
plt.ion()
gifFrames = []
for itr in range(iterations):
    # EVALUATE FITNESS AND FIND THE SWARM'S BEST FLY
    fitnesses = np.array([fitness(X[i]) for i in range(N)])
    s = np.argmin(fitnesses)
    # SHOW THE IMAGE, ALL FLIES (BLUE) AND THE SWARM'S BEST (RED)
    imgplot = plt.imshow(img, cmap='gray')
    for i in range(N):
        plt.gca().add_patch(plt.Circle((X[i][1], X[i][0]), 3, color='b'))
    swarmBestCircle = plt.Circle((X[s][1], X[s][0]), 5, color='r')
    plt.gca().add_patch(swarmBestCircle)
    plt.title(f'Iteration {itr}  best brightness: {-fitnesses[s]:.3f}')
    plt.draw()
    grab_frame(gifFrames)
    display.clear_output(wait=True)
    display.display(plt.gcf())
    time.sleep(.001)
    plt.clf()
    # UPDATE EACH FLY (ELITIST: THE BEST FLY DOES NOT MOVE)
    for i in range(N):
        if i == s:
            continue
        # BEST NEIGHBOUR IN A RING TOPOLOGY
        left = (i - 1) % N
        right = (i + 1) % N
        bNeighbour = left if fitnesses[left] < fitnesses[right] else right
        for d in range(D):
            if np.random.rand() < delta:    # DISTURBANCE: RESTART THIS DIMENSION
                X[i][d] = np.random.randint(lowerB[d], upperB[d] + 1)
                continue
            u = np.random.rand()
            X[i][d] = round(X[bNeighbour][d] + u * (X[s][d] - X[i][d]))
            # OUT OF BOUNDS: RESET TO A RANDOM POSITION
            if X[i][d] < lowerB[d] or X[i][d] > upperB[d]:
                X[i][d] = np.random.randint(lowerB[d], upperB[d] + 1)
save_gif(gifFrames, 'dfo_brightest.gif')
fitnesses = np.array([fitness(X[i]) for i in range(N)])
s = np.argmin(fitnesses)
print('Brightest point found at (row, col):', tuple(X[s]), ' brightness:', -fitnesses[s])

## Advanced tasks

The inputs for tasks 4–7 are in `input_images/` and were made with `make_inputs.py` (run `python make_inputs.py` to regenerate them):

| Task | Input | What it contains |
|---|---|---|
| 4 | `moving.gif` | one bright spot moving along a looping path (150 frames) |
| 5 | `fading.gif` | spot A fades out (frames 30–70) while spot B emerges elsewhere (frames 60–100) |
| 6 | `multi_spot.png` | three equally bright spots |
| 7 | `symmetry.png` | image that is mirror-symmetric about column 300 |

The cell below wraps the DFO update from above into reusable functions, so each task only has to say *what* the fitness is.

In [ ]:
from PIL import ImageSequence
def load_gray(path):
    return np.asarray(Image.open(path).convert('L'), dtype=np.float32) / 255
def load_frames(path, ms=50):
    # PIL MERGES IDENTICAL CONSECUTIVE GIF FRAMES INTO ONE LONGER FRAME, SO EXPAND THEM BY DURATION
    frames = []
    for f in ImageSequence.Iterator(Image.open(path)):
        a = np.asarray(f.convert('L'), dtype=np.float32) / 255
        frames += [a] * max(1, round(f.info.get('duration', ms) / ms))
    return frames
def init_swarm(N, lowerB, upperB):
    return np.random.randint(lowerB, upperB + 1, size=(N, len(lowerB)))
def dfo_step(X, fitnesses, lowerB, upperB, delta, scouts=0):
    # ONE DFO UPDATE OF X (IN PLACE). SAME RULES AS THE DFO BLOCK ABOVE.
    # scouts > 0: THE `scouts` WORST FLIES ARE ALSO MOVED TO RANDOM POSITIONS, SO PART OF THE
    # SWARM KEEPS SEARCHING THE WHOLE IMAGE (NEEDED FOR OBJECTS THAT APPEAR AWAY FROM THE SWARM)
    N, D = X.shape
    s = np.argmin(fitnesses)
    for i in range(N):
        if i == s:
            continue
        left, right = (i - 1) % N, (i + 1) % N
        bNeighbour = left if fitnesses[left] < fitnesses[right] else right
        for d in range(D):
            if np.random.rand() < delta:
                X[i][d] = np.random.randint(lowerB[d], upperB[d] + 1)
                continue
            X[i][d] = round(X[bNeighbour][d] + np.random.rand() * (X[s][d] - X[i][d]))
            if X[i][d] < lowerB[d] or X[i][d] > upperB[d]:
                X[i][d] = np.random.randint(lowerB[d], upperB[d] + 1)
    if scouts > 0:
        for i in np.argsort(fitnesses)[-scouts:]:
            if i != s:
                X[i] = np.random.randint(lowerB, upperB + 1)
def pixel_fitness(frame, x):
    return -frame[x[0]][x[1]]
def show(frame, X, s, title, gifFrames=None):
    plt.imshow(frame, cmap='gray', vmin=0, vmax=1)
    for i in range(len(X)):
        plt.gca().add_patch(plt.Circle((X[i][1], X[i][0]), 3, color='b'))
    plt.gca().add_patch(plt.Circle((X[s][1], X[s][0]), 5, color='r'))
    plt.title(title)
    if gifFrames is not None:
        grab_frame(gifFrames)
    display.clear_output(wait=True)
    display.display(plt.gcf())
    plt.clf()
def track(frames, iterations, N=50, delta=0.001, scouts=0, animate=True, title='', gifName=None):
    # RUN DFO ON A SEQUENCE OF FRAMES (ONE FRAME PER ITERATION).
    # RETURNS THE DISTANCE BETWEEN THE SWARM'S BEST AND THE TRULY BRIGHTEST PIXEL AT EACH ITERATION.
    H, W = frames[0].shape
    lowerB, upperB = np.array([0, 0]), np.array([H - 1, W - 1])
    X = init_swarm(N, lowerB, upperB)
    errors = []
    gifFrames = []
    for itr in range(iterations):
        frame = frames[itr % len(frames)]
        fitnesses = np.array([pixel_fitness(frame, X[i]) for i in range(N)])
        s = np.argmin(fitnesses)
        target = np.unravel_index(frame.argmax(), frame.shape)
        errors.append(np.hypot(X[s][0] - target[0], X[s][1] - target[1]))
        if animate:
            show(frame, X, s, f'{title}  iteration {itr}  error {errors[-1]:.0f}px', gifFrames)
        dfo_step(X, fitnesses, lowerB, upperB, delta, scouts)
    if animate:
        plt.close()
        if gifName:
            save_gif(gifFrames, gifName)
    return np.array(errors)
def compare_deltas(frames, iterations, deltas, scouts=0, runs=10, title=''):
    # MEDIAN TRACKING ERROR OVER SEVERAL RUNS, ONE LINE PER DELTA
    colours = ['#2a78d6', '#eb6834', '#1baf7a']
    plt.figure(figsize=(9, 4))
    for delta, colour in zip(deltas, colours):
        errs = np.array([track(frames, iterations, delta=delta, scouts=scouts, animate=False) for _ in range(runs)])
        plt.plot(np.median(errs, axis=0), color=colour, linewidth=2, label=f'delta = {delta}')
    plt.xlabel('Iteration (= frame)')
    plt.ylabel('Distance to brightest pixel (px)')
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.legend(frameon=False)
    plt.show()

### 4 – Is it possible to trace a moving object?

**Yes.** If the fitness is evaluated on the *current* frame every iteration, the swarm keeps chasing the bright spot. Every fly is re-scored on each new frame, so the "best fly" moves to whichever fly is brightest *now*, and the rest of the swarm follows.

What limits tracking:
- **Speed vs. swarm spread.** DFO tends to collapse the swarm onto one point. If the object moves further in one frame than the swarm's width, nobody lands on it any more and the swarm loses it.
- **Disturbance `delta` keeps the swarm spread out.** A larger `delta` keeps a few flies scattered across the image, so the swarm can pick the object up again after losing it. Too large and the swarm stops converging.

The cell below animates one run on `moving.gif`. The next cell compares the tracking error for different `delta` values (median of 10 runs) at normal speed, and at 5× speed (every 5th frame). At normal speed every `delta` keeps up. At 5× speed, `delta = 0.001` keeps losing the spot, while larger values recover.

In [ ]:
moving = load_frames('input_images/moving.gif')
errors = track(moving, iterations=150, delta=0.01, title='Moving spot', gifName='task4_moving.gif')
print('Median tracking error:', np.median(errors), 'px')

In [ ]:
compare_deltas(moving, 300, deltas=[0.001, 0.01, 0.05], title='Task 4: tracking a moving spot (normal speed)')
compare_deltas(moving[::5], 300, deltas=[0.001, 0.01, 0.05], title='Task 4: tracking a moving spot (5x speed)')

### 5 – How about tracing gradually fading and emerging objects?

This is harder than a moving object. When spot A fades, the swarm has already collapsed around it, so **no fly is near spot B when it appears**. The swarm is only pulled to B once some fly happens to land there. With standard DFO, only the disturbance (`delta`) puts flies in new places.

- With a **small `delta`**, the swarm stays at A's old location long after B is the brightest spot.
- A **larger `delta`** helps a little, but it only moves one coordinate of a fly at a time, so most runs still miss B.

**Fix: scouts.** Each iteration, the few worst flies (here 5 of 50) are moved to completely random positions. These flies are useless where they are anyway. The other 45 flies still converge on the current brightest spot, while the scouts keep sampling the rest of the image. As soon as a scout lands on B and B is brighter than where the swarm sits, that scout becomes the swarm best and the whole swarm moves to B. This is the same idea as the random immigrants used in dynamic optimisation.

In `fading.gif`, between about frames 50 and 80 both spots are dimmer than the background, so the brightest pixel is just a background patch. Expect a large error in that window for every `delta`. What matters is how quickly each run drops back to 0 once B is bright. The first comparison below runs without scouts and the second with 5 scouts.

In [ ]:
fading = load_frames('input_images/fading.gif')
# STANDARD DFO: STAYS WHERE SPOT A WAS AND USUALLY NEVER FINDS SPOT B
errors = track(fading, iterations=150, delta=0.01, title='Fading, no scouts', gifName='task5_fading_no_scouts.gif')
print('Without scouts, final error:', errors[-1], 'px')
# WITH 5 SCOUTS: THE 5 WORST FLIES ARE RANDOMLY RE-PLACED EVERY ITERATION, SO ONE SOON LANDS ON B
errors = track(fading, iterations=150, delta=0.001, scouts=5, title='Fading, 5 scouts', gifName='task5_fading_scouts.gif')
print('With 5 scouts, final error:', errors[-1], 'px')

In [ ]:
compare_deltas(fading, 150, deltas=[0.001, 0.01, 0.05], title='Task 5: no scouts')
compare_deltas(fading, 150, deltas=[0.001, 0.01, 0.05], scouts=5, title='Task 5: 5 scouts per iteration')

### 6 – What if the input image has two or more bright spots?

Standard DFO converges on **one** spot: every fly is pulled towards the single swarm best, so whichever spot is found first wins and the others are ignored (run the first cell below a few times to see it pick different spots).

To find all of them, one simple approach is **sequential search with masking (niching)**:
1. Run DFO and record the best position.
2. Black out a disc around that position in a copy of the image, so it can't be found again.
3. Repeat until the best brightness found falls below a threshold.

Other options: run several independent swarms, or drop the global-best term so flies only follow their local neighbours and the swarm splits into clusters.

In [ ]:
multi = load_gray('input_images/multi_spot.png')
# STANDARD DFO: CONVERGES TO ONE OF THE SPOTS
errors = track([multi], iterations=80, delta=0.001, title='Standard DFO, 3 spots', gifName='task6_standard_dfo.gif')

In [ ]:
def find_bright_spots(image, maxSpots=5, radius=70, minBright=0.8, N=50, iterations=80, delta=0.001, gifName=None):
    H, W = image.shape
    lowerB, upperB = np.array([0, 0]), np.array([H - 1, W - 1])
    rows, cols = np.mgrid[0:H, 0:W]
    work = image.copy()
    spots = []
    gifFrames = []
    for _ in range(maxSpots):
        X = init_swarm(N, lowerB, upperB)
        for itr in range(iterations):
            fitnesses = np.array([pixel_fitness(work, X[i]) for i in range(N)])
            if gifName:
                plt.imshow(work, cmap='gray', vmin=0, vmax=1)
                for i in range(N):
                    plt.gca().add_patch(plt.Circle((X[i][1], X[i][0]), 3, color='b'))
                bestNow = X[np.argmin(fitnesses)]
                plt.gca().add_patch(plt.Circle((bestNow[1], bestNow[0]), 5, color='r'))
                for (r, c) in spots:
                    plt.gca().add_patch(plt.Circle((c, r), 6, color='y'))
                plt.title(f'Search {len(spots) + 1}  iteration {itr}  ({len(spots)} found, yellow)')
                grab_frame(gifFrames)
                plt.clf()
            dfo_step(X, fitnesses, lowerB, upperB, delta)
        fitnesses = np.array([pixel_fitness(work, X[i]) for i in range(N)])
        best = X[np.argmin(fitnesses)]
        if work[best[0]][best[1]] < minBright:
            break
        spots.append(tuple(best))
        work[(rows - best[0]) ** 2 + (cols - best[1]) ** 2 < radius ** 2] = 0    # MASK THIS SPOT
    if gifName:
        plt.close()
        save_gif(gifFrames, gifName)
    return spots, work
spots, masked = find_bright_spots(multi, gifName='task6_find_all_spots.gif')
print('Spots found (row, col):', spots)
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(multi, cmap='gray', vmin=0, vmax=1)
for k, (r, c) in enumerate(spots):
    ax[0].add_patch(plt.Circle((c, r), 6, color='r'))
    ax[0].annotate(str(k + 1), (c + 10, r - 10), color='r', fontsize=12)
ax[0].set_title(f'{len(spots)} spots found (red, in order found)')
ax[1].imshow(masked, cmap='gray', vmin=0, vmax=1)
ax[1].set_title('Image after masking every spot found')
plt.show()

### 7 – If the task was to find symmetry in an image, what would be your fitness function?

To find a **vertical mirror axis**, the search space is the axis column `c` (so `D = 1`). Compare a strip of width `w` to the left of `c` with the mirrored strip to the right:

<𝑠𝑝𝑎𝑛𝑐𝑙𝑎𝑠𝑠="ℎ𝑙𝑗𝑠−𝑐𝑜𝑚𝑚𝑒𝑛𝑡">𝑓(𝑐)=1𝐻𝑤∑𝐻−1𝑟=0∑𝑤𝑘=1∣∣𝑖𝑚𝑔[𝑟][𝑐−𝑘]−𝑖𝑚𝑔[𝑟][𝑐+𝑘]∣∣</𝑠𝑝𝑎𝑛>


`f(c) = 0` means perfect symmetry, so DFO minimises it as before. `c` is bounded to `[w, W-1-w]` so both strips stay inside the image.

Extensions:
- **Horizontal axis:** compare rows instead of columns.
- **Any angle:** add the angle as a second dimension (`D = 2`).
- **Point symmetry:** use `D = 2` for the centre `(r, c)` and compare `img[r+dy][c+dx]` with `img[r-dy][c-dx]`.
- **Flat regions:** a flat region is trivially "symmetric". Dividing by the strip's variance stops plain areas from scoring well.

`symmetry.png` is mirror-symmetric about column 300.

In [ ]:
sym = load_gray('input_images/symmetry.png')
symH, symW = sym.shape
w = 80    # STRIP WIDTH COMPARED ON EACH SIDE OF THE AXIS
def symmetry_fitness(x, image=sym, w=w):
    c = x[0]
    left = image[:, c - w:c]                    # COLUMNS c-w .. c-1
    right = image[:, c + 1:c + w + 1][:, ::-1]  # COLUMNS c+w .. c+1 (MIRRORED)
    return np.mean(np.abs(left - right))
N, iterations, delta = 20, 40, 0.001
lowerB, upperB = np.array([w]), np.array([symW - 1 - w])
X = init_swarm(N, lowerB, upperB)
gifFrames = []
for itr in range(iterations):
    fitnesses = np.array([symmetry_fitness(X[i]) for i in range(N)])
    s = np.argmin(fitnesses)
    plt.imshow(sym, cmap='gray', vmin=0, vmax=1)
    for i in range(N):
        plt.axvline(X[i][0], color='b', alpha=0.4, linewidth=1)
    plt.axvline(X[s][0], color='r', linewidth=2)
    plt.title(f'Iteration {itr}  axis at column {X[s][0]}  f = {fitnesses[s]:.4f}')
    grab_frame(gifFrames)
    display.clear_output(wait=True)
    display.display(plt.gcf())
    plt.clf()
    dfo_step(X, fitnesses, lowerB, upperB, delta)
plt.close()
save_gif(gifFrames, 'task7_symmetry.gif')
fitnesses = np.array([symmetry_fitness(X[i]) for i in range(N)])
bestC = X[np.argmin(fitnesses)][0]
print('Symmetry axis found at column', bestC)

In [ ]:
# CHECK: THE WHOLE FITNESS LANDSCAPE (BRUTE FORCE) VS. WHAT DFO FOUND
cs = np.arange(lowerB[0], upperB[0] + 1)
landscape = [symmetry_fitness([c]) for c in cs]
plt.figure(figsize=(9, 4))
plt.plot(cs, landscape, color='#2a78d6', linewidth=2)
plt.axvline(bestC, color='#eb6834', linewidth=2, linestyle='--')
plt.annotate(f'DFO result: column {bestC}', (bestC + 5, max(landscape) * 0.9), color='#52514e')
plt.xlabel('Axis column c')
plt.ylabel('f(c)  (lower = more symmetric)')
plt.title('Task 7: symmetry fitness for every possible axis')
plt.grid(alpha=0.3)
plt.show()